In [0]:
import os, sys
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import bronze as br
from src import meta

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_dates", "2026-11-24,2026-11-25,2026-11-26", "Load dates (comma separated)")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")
dbutils.widgets.text("run_id", "", "Run id (blank = generate)")
dbutils.widgets.text("pipeline_name", "manual", "Pipeline name")

resource_param = dbutils.widgets.get("resource").strip()
load_dates = [d.strip() for d in dbutils.widgets.get("load_dates").split(",") if d.strip()]
catalog_override = dbutils.widgets.get("catalog").strip()
run_id = dbutils.widgets.get("run_id").strip() or cl.new_batch_id()
pipeline_name = dbutils.widgets.get("pipeline_name").strip()

overrides = {"project": {"catalog": catalog_override}} if catalog_override else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if resource_param.upper() == "ALL" else [resource_param]
resources = cl.get_resources(cfg, only)
print(f"run_id={run_id} | dates={load_dates}")
print("Resources:", [r["name"] for r in resources])

In [0]:
meta.create_meta_tables(spark, cfg)

results = []
for r in resources:                       # Patient -> Encounter -> Observation -> Condition
    table = cl.resource_table(cfg, "bronze", r["name"])
    for d in load_dates:
        t0 = cl.utc_now()
        batch_id, expected, actual, status, error = None, 0, 0, "SUCCESS", None
        try:
            df, manifest = br.build_bronze_df(spark, cfg, r["name"], d)
            if df is None:
                status = "NO_RAW_DATA"
            else:
                batch_id = manifest.select("batch_id").first()[0]
                expected = int(br.expected_record_count(manifest))
                br.write_bronze(spark, df, table, d)
                actual = br.bronze_count(spark, table, d)
                if actual != expected:
                    status, error = "FAILED", f"count mismatch: expected {expected}, got {actual}"
        except Exception as e:
            status, error = "FAILED", f"{type(e).__name__}: {e}"
        t1 = cl.utc_now()

        meta.log_run(spark, cfg, run_id=run_id, pipeline_name=pipeline_name, stage="BRONZE",
                     resource_type=r["name"], load_date=d, status=status, batch_id=batch_id,
                     source_format="json", started_at=t0, ended_at=t1,
                     data_saved_at=t1 if status == "SUCCESS" else None,
                     records_read=expected, records_written=actual, error_message=error)
        results.append({"resource": r["name"], "load_date": d, "expected": expected,
                        "bronze_rows": actual, "status": status, "error": error})

res_df = pd.DataFrame(results)
display(res_df)

bad = res_df[res_df["status"] != "SUCCESS"]
if len(bad):
    raise Exception(f"Bronze load problems:\n{bad.to_string()}")

In [0]:
t = cl.resource_table(cfg, "bronze", "Patient")
display(spark.sql(f"""
  SELECT resource_id, meta_version_id, meta_last_updated, extraction_timestamp,
         api_url_or_params, raw_saved_at, bronze_loaded_at, load_date, page_number,
         substring(resource_json, 1, 120) AS resource_json_preview
  FROM {t} ORDER BY load_date, page_number LIMIT 10
"""))

display(spark.sql(f"SELECT load_date, count(*) AS rows, count(DISTINCT resource_id) AS distinct_ids FROM {t} GROUP BY load_date ORDER BY load_date"))
display(spark.sql(f"DESCRIBE HISTORY {t}").select("version", "timestamp", "operation", "operationParameters"))

In [0]:
log = cl.meta_table(cfg, "run_log")

# Everything this run did, in order
display(spark.sql(f"""
  SELECT stage, resource_type, load_date, status, pages_fetched, records_read,
         records_written, truncated, started_at, ended_at, data_saved_at, error_message
  FROM {log} WHERE run_id = '{run_id}' ORDER BY started_at
"""))

# Raw vs Bronze reconciliation
display(spark.sql(f"""
  SELECT r.resource_type, r.load_date, r.records_written AS raw_records,
         b.records_written AS bronze_rows, r.truncated,
         (r.records_written = b.records_written) AS counts_match
  FROM (SELECT * FROM {log} WHERE stage='RAW'    AND run_id='{run_id}') r
  JOIN (SELECT * FROM {log} WHERE stage='BRONZE' AND run_id='{run_id}') b
    ON r.resource_type = b.resource_type AND r.load_date = b.load_date
  ORDER BY r.resource_type, r.load_date
"""))